# 04 - Comandos DML em Tabelas Delta Lake

Demonstra operações DML (**INSERT**, **UPDATE**, **DELETE**) em tabelas Delta Lake armazenadas no MinIO, além de recursos como **HISTORY** e **TIME TRAVEL**.

**Pré-requisitos:** Notebook `03` executado (tabelas Delta no bucket `bronze`).

## 1. Configuração e SparkSession

In [1]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta import *
from delta.tables import DeltaTable

load_dotenv(override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

spark = (
    SparkSession.builder
    .appName('DML_Delta_Lake')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada com sucesso!')
spark

26/05/04 19:44:52 WARN Utils: Your hostname, DESKTOP-C71TG2N resolves to a loopback address: 127.0.1.1; using 172.22.160.68 instead (on interface eth0)
26/05/04 19:44:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/ian/spark-delta-minio-sqlserver/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ian/.ivy2/cache
The jars for the packages stored in: /home/ian/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4eda2403-5acf-405c-bd85-ae5bcde4aa47;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 1626ms :: artifacts dl 37ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hadoop#ha

SparkSession criada com sucesso!


## 2. Registrar Tabelas Delta como SQL Tables

In [2]:
# Registrar as tabelas Delta Lake para uso com Spark SQL
tabelas_delta = ['categorias', 'produtos', 'clientes', 'vendas', 'itens_venda']

for tabela in tabelas_delta:
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tabela}
        USING delta
        LOCATION '{delta_path}'
    """)

# Listar tabelas registradas
print('Tabelas registradas no Spark:')
spark.sql('SHOW TABLES').show(truncate=False)

26/05/04 19:45:20 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/05/04 19:45:23 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

Tabelas registradas no Spark:
+---------+-----------+-----------+
|namespace|tableName  |isTemporary|
+---------+-----------+-----------+
|default  |categorias |false      |
|default  |clientes   |false      |
|default  |itens_venda|false      |
|default  |produtos   |false      |
|default  |vendas     |false      |
+---------+-----------+-----------+



## 3. Consultar Dados Atuais (SELECT)

In [3]:
# Visualizar o estado inicial das tabelas do e-commerce
print('=== CATEGORIAS ===')
spark.sql('SELECT * FROM categorias ORDER BY id_categoria').show()

print('=== PRODUTOS ORIGINAIS (Primeiros 10) ===')
spark.sql('SELECT * FROM produtos ORDER BY id_produto LIMIT 10').show()

print('=== CLIENTES ORIGINAIS (Primeiros 10) ===')
spark.sql('SELECT * FROM clientes ORDER BY id_cliente LIMIT 10').show()

=== CATEGORIAS ===


26/05/04 19:45:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+------------+--------------+--------------------+
|id_categoria|nome_categoria|           descricao|
+------------+--------------+--------------------+
|           1|   Eletrônicos|Smartphones, note...|
|           2|     Vestuário|Roupas masculinas...|
|           3|        Livros|Ficção, técnicos ...|
|           4|          Casa|Móveis e itens de...|
|           5|      Esportes|Artigos esportivo...|
+------------+--------------+--------------------+

=== PRODUTOS ORIGINAIS (Primeiros 10) ===


+----------+-------------------+------------+------+-------+
|id_produto|               nome|id_categoria| preco|estoque|
+----------+-------------------+------------+------+-------+
|         1|     Smartphone XYZ|           1|1500.0|     50|
|         2|       Notebook Pro|           1|3500.0|     30|
|         3|    Camiseta Básica|           2|  50.0|    100|
|         4|        Calça Jeans|           2| 120.0|     80|
|         5| O Senhor dos Anéis|           3|  60.0|     40|
|         6|Livro Spark e Delta|           3|  85.0|     60|
|         7|      Sofá Retrátil|           4|1200.0|     10|
|         8|     Mesa de Jantar|           4| 800.0|     15|
|         9|    Bola de Futebol|           5|  80.0|     50|
|        10|   Tênis de Corrida|           5| 250.0|     40|
+----------+-------------------+------------+------+-------+

=== CLIENTES ORIGINAIS (Primeiros 10) ===


+----------+--------------+------+------------+
|id_cliente|          nome|estado|status_conta|
+----------+--------------+------+------------+
|         1|    João Silva|    SP|       Ativo|
|         2|Maria Oliveira|    RJ|       Ativo|
|         3| Carlos Santos|    MG|     Inativo|
|         4|     Ana Costa|    BA|       Ativo|
|         5|   Pedro Alves|    PR|       Ativo|
|         6| Fernanda Lima|    RS|     Inativo|
|         7|   Lucas Gomes|    PE|       Ativo|
|         8| Juliana Rocha|    SC|       Ativo|
|         9|   Marcos Dias|    CE|       Ativo|
|        10| Camila Mendes|    GO|       Ativo|
+----------+--------------+------+------------+



In [4]:
# Contagem de registros por tabela
print(f'{"Tabela":<15} {"Registros":>10}')
print('-' * 27)
for tabela in tabelas_delta:
    count = spark.sql(f'SELECT COUNT(*) as cnt FROM {tabela}').collect()[0]['cnt']
    print(f'{tabela:<15} {count:>10}')

Tabela           Registros
---------------------------


categorias               5


produtos                10


clientes                10


vendas                  10


[Stage 55:===========================================>            (39 + 8) / 50]

itens_venda             13


---
## 4. INSERT - Inserir Novos Registros

Vamos inserir novos registros nas tabelas `Categorias`, `Produtos` e `Clientes`.

In [5]:
print("--- INSERT: Novas Categorias, Produtos e Clientes ---")

# 1. Inserindo na tabela categorias
spark.sql("""
    INSERT INTO categorias VALUES
    (6, 'Games', 'Consoles e Jogos digitais')
""")

# 2. Inserindo na tabela produtos (vinculados à categoria 6)
spark.sql("""
    INSERT INTO produtos VALUES
    (11, 'PlayStation 5', 6, 4500.00, 20),
    (12, 'Controle DualSense', 6, 450.00, 50)
""")

# 3. Inserindo na tabela clientes
spark.sql("""
    INSERT INTO clientes VALUES
    (11, 'Roberto Carlos', 'SP', 'Ativo'),
    (12, 'Aline Barros', 'RJ', 'Ativo')
""")

print("Registros inseridos com sucesso nas 3 tabelas!\n")
spark.sql("SELECT * FROM categorias WHERE id_categoria = 6").show()
spark.sql("SELECT * FROM produtos WHERE id_produto >= 11 ORDER BY id_produto").show()
spark.sql("SELECT * FROM clientes WHERE id_cliente >= 11 ORDER BY id_cliente").show()

--- INSERT: Novas Categorias, Produtos e Clientes ---


Registros inseridos com sucesso nas 3 tabelas!



+------------+--------------+--------------------+
|id_categoria|nome_categoria|           descricao|
+------------+--------------+--------------------+
|           6|         Games|Consoles e Jogos ...|
+------------+--------------+--------------------+



+----------+------------------+------------+------+-------+
|id_produto|              nome|id_categoria| preco|estoque|
+----------+------------------+------------+------+-------+
|        11|     PlayStation 5|           6|4500.0|     20|
|        12|Controle DualSense|           6| 450.0|     50|
+----------+------------------+------------+------+-------+



+----------+--------------+------+------------+
|id_cliente|          nome|estado|status_conta|
+----------+--------------+------+------------+
|        11|Roberto Carlos|    SP|       Ativo|
|        12|  Aline Barros|    RJ|       Ativo|
+----------+--------------+------+------------+



---
## 5. UPDATE - Atualizar Registros

Vamos atualizar registros existentes nas tabelas.

In [6]:
from pyspark.sql.functions import lit

print("--- UPDATE: Atualizando dados nas 3 tabelas ---")

# 1. UPDATE via SQL: Atualizando categorias (melhorando a descrição)
spark.sql("""
    UPDATE categorias 
    SET descricao = 'Consoles, Jogos e Acessórios' 
    WHERE id_categoria = 6
""")

# 2. UPDATE via DeltaTable API: Reduzindo o preço do PS5
dt_produtos = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/produtos')
dt_produtos.update(
    condition="id_produto = 11",
    set={"preco": lit(4299.90)}
)

# 3. UPDATE via SQL: Atualizando clientes (mudando status e estado)
spark.sql("""
    UPDATE clientes 
    SET status_conta = 'Inativo', estado = 'MG' 
    WHERE id_cliente = 11
""")

print("Registros atualizados com sucesso!\n")
spark.sql("SELECT * FROM categorias WHERE id_categoria = 6").show()
spark.sql("SELECT * FROM produtos WHERE id_produto = 11").show()
spark.sql("SELECT * FROM clientes WHERE id_cliente = 11").show()

--- UPDATE: Atualizando dados nas 3 tabelas ---


Registros atualizados com sucesso!



+------------+--------------+--------------------+
|id_categoria|nome_categoria|           descricao|
+------------+--------------+--------------------+
|           6|         Games|Consoles, Jogos e...|
+------------+--------------+--------------------+



+----------+-------------+------------+------+-------+
|id_produto|         nome|id_categoria| preco|estoque|
+----------+-------------+------------+------+-------+
|        11|PlayStation 5|           6|4299.9|     20|
+----------+-------------+------------+------+-------+



+----------+--------------+------+------------+
|id_cliente|          nome|estado|status_conta|
+----------+--------------+------+------------+
|        11|Roberto Carlos|    MG|     Inativo|
+----------+--------------+------+------------+



---
## 6. DELETE - Remover Registros

Vamos deletar registros de tabelas Delta.

In [7]:
print("--- DELETE: Removendo registros das 3 tabelas ---")

# 1. DELETE via DeltaTable API: Deletando a categoria criada
dt_categorias = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/categorias')
dt_categorias.delete("id_categoria = 6")

# 2. DELETE via SQL: Deletando um dos produtos (Controle DualSense)
spark.sql("""
    DELETE FROM produtos 
    WHERE id_produto = 12
""")

# 3. DELETE via SQL: Deletando um dos clientes (Aline Barros)
spark.sql("""
    DELETE FROM clientes 
    WHERE id_cliente = 12
""")

print("Registros removidos com sucesso!\n")
print("Categoria 6:")
spark.sql("SELECT * FROM categorias WHERE id_categoria = 6").show() # Retornará vazio

print("Produtos restantes (ID >= 11):")
spark.sql("SELECT * FROM produtos WHERE id_produto >= 11").show() # Só deve sobrar o PS5 (11)

print("Clientes restantes (ID >= 11):")
spark.sql("SELECT * FROM clientes WHERE id_cliente >= 11").show() # Só deve sobrar o Roberto (11)

--- DELETE: Removendo registros das 3 tabelas ---


Registros removidos com sucesso!

Categoria 6:


+------------+--------------+---------+
|id_categoria|nome_categoria|descricao|
+------------+--------------+---------+
+------------+--------------+---------+

Produtos restantes (ID >= 11):


+----------+-------------+------------+------+-------+
|id_produto|         nome|id_categoria| preco|estoque|
+----------+-------------+------------+------+-------+
|        11|PlayStation 5|           6|4299.9|     20|
+----------+-------------+------------+------+-------+

Clientes restantes (ID >= 11):


+----------+--------------+------+------------+
|id_cliente|          nome|estado|status_conta|
+----------+--------------+------+------------+
|        11|Roberto Carlos|    MG|     Inativo|
+----------+--------------+------+------------+



---
## 7. HISTORY - Histórico de Versões Delta

O Delta Lake mantém um log de transações que permite visualizar o histórico completo de alterações.

In [8]:
# Histórico da tabela Categorias (onde usamos DeltaTable API no Delete)
print('=== HISTÓRICO DA TABELA: CATEGORIAS ===')
dt_categorias = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/categorias')
dt_categorias.history().select('version', 'timestamp', 'operation').show(truncate=False)

# Histórico da tabela Produtos (onde usamos DeltaTable API no Update)
print('\n=== HISTÓRICO DA TABELA: PRODUTOS ===')
spark.sql('DESCRIBE HISTORY produtos').select('version', 'timestamp', 'operation').show(truncate=False)

=== HISTÓRICO DA TABELA: CATEGORIAS ===
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|3      |2026-05-04 19:47:05|DELETE   |
|2      |2026-05-04 19:46:42|UPDATE   |
|1      |2026-05-04 19:46:23|WRITE    |
|0      |2026-05-04 19:43:16|WRITE    |
+-------+-------------------+---------+


=== HISTÓRICO DA TABELA: PRODUTOS ===
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|3      |2026-05-04 19:47:10|DELETE   |
|2      |2026-05-04 19:46:46|UPDATE   |
|1      |2026-05-04 19:46:24|WRITE    |
|0      |2026-05-04 19:43:26|WRITE    |
+-------+-------------------+---------+



---
## 8. TIME TRAVEL - Viagem no Tempo

O Delta Lake permite ler versões anteriores dos dados.

In [9]:
print('=== TIME TRAVEL: Tabela de Clientes (Versão 0 vs Atual) ===')

# Lendo a versão "0" (estado original logo após a ingestão dos CSVs)
df_cliente_v0 = spark.read.format('delta').option('versionAsOf', 0).load(f's3a://{BRONZE_BUCKET}/clientes')

# Lendo a versão atual (com todas as modificações DML aplicadas)
df_cliente_atual = spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/clientes')

print(f'Versão 0 (Original): {df_cliente_v0.count()} registros')
print(f'Versão Atual:        {df_cliente_atual.count()} registros')

print('\nRegistros ADICIONADOS ou MODIFICADOS em relação à versão original:')
# Subtrai o dataframe original do atual para ver o que sobrou de diferente
df_cliente_atual.subtract(df_cliente_v0).show()

=== TIME TRAVEL: Tabela de Clientes (Versão 0 vs Atual) ===


Versão 0 (Original): 10 registros
Versão Atual:        11 registros

Registros ADICIONADOS ou MODIFICADOS em relação à versão original:


+----------+--------------+------+------------+
|id_cliente|          nome|estado|status_conta|
+----------+--------------+------+------------+
|        11|Roberto Carlos|    MG|     Inativo|
+----------+--------------+------+------------+



---
## 9. Resumo Final

In [10]:
print('=' * 70)
print('RESUMO DAS OPERAÇÕES DML NO DELTA LAKE')
print('=' * 70)
print()
print('1. INSERT (Via Spark SQL):')
print('   - Nova categoria 6 (Games)')
print('   - Novos produtos 11 (PS5) e 12 (Controle)')
print('   - Novos clientes 11 (Roberto) e 12 (Aline)')
print()
print('2. UPDATE (Mix de Tecnologias):')
print('   - Categoria 6: Descrição alterada (Spark SQL)')
print('   - Produto 11: Preço do PS5 reduzido (DeltaTable API)')
print('   - Cliente 11: Status e estado atualizados (Spark SQL)')
print()
print('3. DELETE (Mix de Tecnologias):')
print('   - Categoria 6: Removida (DeltaTable API)')
print('   - Produto 12: Removido (Spark SQL)')
print('   - Cliente 12: Removida (Spark SQL)')
print()
print('4. HISTORY e TIME TRAVEL:')
print('   - Comprovada a rastreabilidade e versionamento do Delta Lake,')
print('     independentemente de usar SQL puro ou a API do Python.')
print('=' * 70)

# Encerrando a sessão de forma segura
spark.stop()
print('\nSparkSession finalizada com sucesso. Pipeline concluído!')

RESUMO DAS OPERAÇÕES DML NO DELTA LAKE

1. INSERT (Via Spark SQL):
   - Nova categoria 6 (Games)
   - Novos produtos 11 (PS5) e 12 (Controle)
   - Novos clientes 11 (Roberto) e 12 (Aline)

2. UPDATE (Mix de Tecnologias):
   - Categoria 6: Descrição alterada (Spark SQL)
   - Produto 11: Preço do PS5 reduzido (DeltaTable API)
   - Cliente 11: Status e estado atualizados (Spark SQL)

3. DELETE (Mix de Tecnologias):
   - Categoria 6: Removida (DeltaTable API)
   - Produto 12: Removido (Spark SQL)
   - Cliente 12: Removida (Spark SQL)

4. HISTORY e TIME TRAVEL:
   - Comprovada a rastreabilidade e versionamento do Delta Lake,
     independentemente de usar SQL puro ou a API do Python.

SparkSession finalizada com sucesso. Pipeline concluído!
